[<< Sommaire QC](../README.md) | [Précédent : QC-Py-43-Diffusion-Series-Synthetiques <<](./QC-Py-43-Diffusion-Series-Synthetiques.ipynb) | [Suivant : QC-Py-45-Noyau-Risque-Deterministe >>](./QC-Py-45-Noyau-Risque-Deterministe.ipynb)

# QC-Py-44 — Agent de trading multi-LLM : analystes, débat et mémoire

**Série QuantConnect / Python — volet P2 de l'EPIC #19306.**

Un LLM seul répond à une question. Une **firme de trading** fait autre chose : des analystes
travaillent des angles différents, un débat confronte les thèses, un gérant tranche, et surtout
**la firme se souvient** de ce que ses décisions passées ont produit. Le cadre
[TradingAgents](https://arxiv.org/abs/2412.20138) (Xiao, Sun, Luo & Wang, arXiv:2412.20138)
reproduit cette organisation avec des agents LLM ; ce carnet en reprend l'ossature à l'échelle
pédagogique, et mesure ce qu'elle apporte — ou n'apporte pas.

## Ce que ce carnet mesure

1. Le **cadre complet** : deux analystes, un débat bull/bear borné, un gérant (PM), et une
   **mémoire de décisions soldées** — chaque décision est journalisée, soldée au terme de
   l'horizon de détention contre le rendement réalisé, puis résumée en une phrase réinjectée
   dans les décisions suivantes.
2. Le **témoin négatif** : le même modèle, le même contexte numérique, **un seul appel**. Si le
   débat et la mémoire n'ajoutent rien, les deux bras doivent être indiscernables.
3. Le **verdict** de la doctrine §C du dépôt : quatre graines, pertes appariées,
   Diebold-Mariano, frais de transaction, et un verdict explicite — jamais « prometteur ».
4. La **reproductibilité** : chaque appel au modèle est inscrit dans un journal, qui permet de
   rejouer la campagne sans aucun appel (rejeu strict) ou de ne rappeler que ce qu'une
   consigne modifiée change (rejeu mixte) ; sur les décisions journalisées, un **noyau de
   risque déterministe** se greffe et se juge à coût nul.

## Ce que ce carnet n'est pas

Ce n'est pas une réplication de TradingAgents : le cadre original fait intervenir des
analystes fondamental, sentiment et technique sur des **actualités**, une équipe de risque et
un gestionnaire de portefeuille multi-actifs, et il est évalué sur des actions américaines.
Ici, deux analystes travaillent **les mêmes chiffres** sous deux lentilles différentes
(momentum et risque), sur **un seul actif** liquide, sans données textuelles. La conclusion
transférable est méthodologique : ce qui se mesure, c'est l'effet du protocole de délibération
et de la mémoire — pas la performance d'un système de production.

## 1. Le cadre, les rôles, et les organes natifs du dépôt

| Rôle | Ce qu'il reçoit | Ce qu'il rend |
|---|---|---|
| **Analyste momentum** | les indicateurs de la date de décision | biais, prévision chiffrée, argument |
| **Analyste risque** | les mêmes indicateurs, lentille réversion/volatilité | biais, prévision chiffrée, argument |
| **Bull** et **Bear** | le contexte **et** les deux analyses | un débat borné à deux tours |
| **PM** | contexte, analyses, débat, **mémoire soldée** | décision `LONG`/`FLAT`/`SHORT` + prévision |
| **Réflexion** | décision, prévision, réalisé | une leçon d'une phrase, stockée |

**Organes natifs (règle organ-first).** Le débat n'est pas réécrit ici : il passe par
`AgentGroupChat` de la série `GenAI/SemanticKernel/03-SemanticKernel-Agents.ipynb`, avec une
stratégie de sélection séquentielle et une borne de deux tours. Les rôles hors débat sont des
appels de complétion simples, comme dans `QC-Py-42`. Le **règlement** ne dépend d'aucune
opinion du modèle : il se fait sur les rendements réalisés du panier local, et le
Diebold-Mariano est le même code que celui de `QC-Py-42`.

**La question falsifiable.** Le cadre vaut-il mieux qu'un appel unique au même modèle sur le
même contexte ? Si le débat et la mémoire n'apportent rien, l'écart de perte entre les deux
bras doit être indistinguishable de zéro — et le verdict doit le dire.

In [1]:
import os, json, re, time, math, warnings
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)

APPELS = {"n": 0}            # compteur d'appels reels, imprime en fin de campagne

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY absente de l'environnement : le carnet exige des appels LLM reels "
        "(C.2). Aucune reponse simulee n'est committable. Exportez la cle puis relancez.")
print("OPENAI_API_KEY presente :", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY presente : True


In [2]:
SEEDS = (0, 1, 7, 42)        # plancher multi-seed de la doctrine §C
HORIZON = 5                  # jours de detention, en seances
COST_BPS = 10                # frais par cote (crypto), en points de base
N_POINTS = 8                 # points de decision par graine
ESPACEMENT = 120             # jours calendaires entre deux points de decision
                             # (8 points couvrent ~2,3 ans : plusieurs regimes, pas un seul)
ACTIF = "BTC-USD"

MODELE = "gpt-5.6-luna"      # meme modele que QC-Py-26 et QC-Py-42 : la serie reste homogene
MAX_TOKENS = 260

print("modele   :", MODELE)
print("graines  :", SEEDS, "| horizon :", HORIZON, "seances | points :", N_POINTS)

modele   : gpt-5.6-luna
graines  : (0, 1, 7, 42) | horizon : 5 seances | points : 8


### Lecture — le design d'expérience en une ligne

`modele: gpt-5.6-luna, graines (0,1,7,42), horizon 5 seances, points 8`. Chaque champ de cette configuration est une décision expérimentale qui se paiera plus loin :

- **Quatre graines standard `(0,1,7,42)`** : le protocole multi-graines du dépôt en exige au moins 4. Ce sont les graines de la *campagne* (échantillonnage des points de décision et du débat), pas d'un entraînement — il n'y a rien à entraîner ici, le modèle est figé ; ce qui varie entre graines, c'est le plan d'expérience qui sollicite le LLM. La dispersion inter-graines mesurée en section 8 (`sigma 0.1874`) est donc la variance du *dispositif*, pas celle d'un apprentissage. Les graines fixent le plan, pas les réponses du modèle : la section 10 mesure ce qui varie encore d'une exécution à l'autre.
- **`horizon 5 seances`** : chaque décision engage une position tenue 5 séances — ni le jour (bruit pur), ni le trimestre (le contexte de marché aura changé). L'exercice 3 fera varier ce paramètre (1, 5, 20) parce qu'il structure tout : la marge de soldage des positions, la granularité des frais (un aller-retour de 20 bps par décision), et la nature même de ce que le cadre « décide ».
- **`points 8`** : le nombre de contextes par graine. Avec 4 graines, le carnet produira 32 paires décision/rendement — le dénominateur de tous les tests de la section 8. C'est volontairement petit : un plan d'expérience LLM se paie en appels (9 par décision, ~288 pour la campagne), et la question du carnet est précisément de mesurer si une campagne modeste suffit à distinguer un edge du bruit.

## 2. Les données

Le panier crypto local du dépôt (`datasets/yfinance/crypto_panier/`) sert de support : il est
versionné, donc le carnet ne dépend d'aucun accès réseau à l'exécution. On travaille sur un
seul actif — `BTC-USD` — parce que la décision porte sur une direction et une amplitude, et
que multiplier les actifs multiplierait le coût des appels sans changer la question posée.

In [3]:
def trouver_panier():
    ici = os.path.abspath(os.getcwd())
    for _ in range(7):
        cand = os.path.join(ici, "MyIA.AI.Notebooks", "QuantConnect",
                            "datasets", "yfinance", "crypto_panier")
        if os.path.isdir(cand):
            return cand
        ici = os.path.dirname(ici)
    raise FileNotFoundError("panier introuvable depuis %s" % os.getcwd())


PANIER = trouver_panier()
_rel = PANIER[PANIER.find("MyIA.AI.Notebooks"):] if "MyIA.AI.Notebooks" in PANIER else PANIER
fichier = os.path.join(PANIER, "%s_2018-01-01_2026-05-01.csv" % ACTIF)
brut = pd.read_csv(fichier, parse_dates=["Date"]).set_index("Date").sort_index()
prix = brut["Close"].astype(float).dropna()
rend = np.log(prix).diff().dropna()

print("panier     :", _rel)
print("actif      :", ACTIF)
print("periode    :", prix.index.min().date(), "->", prix.index.max().date())
print("seances    :", len(prix), "| rendements log :", len(rend))

panier     : MyIA.AI.Notebooks\QuantConnect\datasets\yfinance\crypto_panier
actif      : BTC-USD
periode    : 2018-01-01 -> 2026-04-30
seances    : 3042 | rendements log : 3041


### Lecture — 3 042 séances, une seule perte d'information

`BTC-USD, 2018-01-01 -> 2026-04-30, seances 3042, rendements log 3041`. Trois vérifications se lisent d'un coup :

1. **3 041 = 3 042 − 1, exactement.** Le premier rendement n'existe pas : il faudrait une séance avant le 2018-01-01 pour le calculer. Chaque transformation « n observations → n−1 rendements » perd la même ligne — le vérifier ici, c'est savoir relire n'importe quel carnet de séries temporelles.
2. **Des log-rendements, pas des rendements simples.** La série porte l'additivité temporelle : le rendement cumulé sur k séances est la *somme* des k log-rendements, sans produit composé. C'est le choix standard pour mesurer des edges par fenêtre — la moyenne des log-rendements d'une position se compare directement à sa longueur, ce que les rendements simples ne permettent pas.
3. **La fenêtre couvre trois régimes de marché au moins.** 2018 (fin de cycle), 2020-2021 (expansion), 2022 (effondrement LUNA/FTX), 2024-2025 (reprise). Les 8 points de décision échantillonneront dans ce stock — la dispersion de 4,70 pts mesurée à la cellule suivante en est la conséquence directe : des fenêtres prises dans des régimes différents ne peuvent pas se ressembler.

In [4]:
def contexte(indice):
    # Tout ce que les agents voient : des nombres calcules, jamais une opinion.
    p = prix.iloc[indice]
    r20 = prix.iloc[indice] / prix.iloc[indice - 20] - 1.0
    v20 = rend.iloc[indice - 20:indice].std() * np.sqrt(365)
    haut60 = prix.iloc[indice - 60:indice + 1].max()
    dd = p / haut60 - 1.0
    ma20 = prix.iloc[indice - 20:indice].mean()
    z = (p - ma20) / prix.iloc[indice - 20:indice].std()
    sk = rend.iloc[indice - 60:indice].skew()
    cinq = rend.iloc[indice - 5:indice]
    return {
        "date": str(prix.index[indice].date()),
        "cloture": float(p),
        "ret20": float(r20),
        "vol20": float(v20),
        "drawdown60": float(dd),
        "zscore20": float(z),
        "skew60": float(sk),
        "cinq_jours": [round(float(x) * 100, 2) for x in cinq],
        "realise_5j": float(prix.iloc[indice + HORIZON] / p - 1.0),
    }


# Points de decision : espaces regulierement, du plus ancien au plus recent, en gardant
# la marge de HORIZON seances pour que chaque decision soit soldable.
dernier = len(prix) - 1 - HORIZON
INDICES = [dernier - ESPACEMENT * k for k in range(N_POINTS - 1, -1, -1)]
CTX = [contexte(i) for i in INDICES]

table = pd.DataFrame([{
    "date": c["date"],
    "ret20 %": round(c["ret20"] * 100, 1),
    "vol20 %": round(c["vol20"] * 100, 1),
    "dd60 %": round(c["drawdown60"] * 100, 1),
    "z20": round(c["zscore20"], 2),
    "realise 5j %": round(c["realise_5j"] * 100, 2),
} for c in CTX])
print("points de decision :", len(CTX))
print("dispersion des rendements realises a 5 j : ecart-type %.2f pts"
      % (table["realise 5j %"].std()))
table

points de decision : 8
dispersion des rendements realises a 5 j : ecart-type 4.70 pts


,date,ret20 %,vol20 %,dd60 %,z20,realise 5j %
0,2024-01-06,6.3,42.7,-2.2,0.87,5.41
1,2024-05-05,1.0,52.1,-12.4,0.33,-5.06
2,2024-09-02,-2.5,50.2,-13.4,-0.48,-8.41
3,2024-12-31,-7.7,43.2,-12.0,-1.29,5.23
4,2025-04-30,18.3,37.8,-0.8,1.15,0.57
5,2025-08-28,-3.6,38.7,-8.8,-1.04,-1.19
6,2025-12-26,-2.2,28.2,-23.5,-0.77,0.24
7,2026-04-25,12.5,40.6,-0.8,1.25,-1.68


### Lecture — huit fenêtres de décision, un seul marché

`points de decision 8, dispersion ecart-type 4.70 pts`. Deux nombres, deux rôles différents :

- **8 points de décision** fixent le budget expérimental : 8 contextes par graine, 32 paires au total avec 4 graines. Ce n'est pas un échantillonnage exhaustif du marché — c'est un plan d'expérience : chaque point ouvre une fenêtre de 5 séances (l'horizon de la section 1) pendant laquelle le cadre détient une position.
- **σ = 4,70 pts de dispersion** décrit la variété des contextes offerts : les 8 points ne sont pas 8 copies du même marché. Une dispersion de presque 5 points entre fenêtres garantit que le cadre est confronté à des régimes différents — c'est une condition minimale pour que la question « le cadre s'adapte-t-il au contexte ? » ait un sens. Avec σ proche de zéro, les 8 décisions mesureraient une seule situation répétée 8 fois, et le verdict de la section 7 porterait sur une coïncidence de fenêtre (la question exacte que l'exercice 3 généralise aux horizons).

Ces deux nombres sont la maille du filet : assez de dispersion pour que les contextes diffèrent, trop peu de points pour qu'un edge modeste soit significatif — la tension que le verdict INCONCLUSIVE mesurera sans fard.

## 3. Les rôles hors débat

Les analystes et le PM sont des appels de complétion simples. Deux précautions valent d'être
posées ici :

- **Le schéma de sortie est imposé.** Chaque rôle rend un JSON à clés fixes, extrait par une
  fonction tolérante, et une réponse non parseable est **comptée** puis remplacée par une
  décision neutre (`FLAT`, prévision 0) — jamais par une valeur inventée qui aurait l'air
  d'une réponse du modèle.
- **Aucun repli simulé.** Si l'API échoue après reprises, le carnet lève : une sortie simulée
  ne se commite pas (C.2).

In [5]:
import hashlib
import tempfile
from collections import Counter
from pathlib import Path

from openai import OpenAI

client = OpenAI()          # lit OPENAI_API_KEY dans l'environnement


class Journal:
    """Trace d'execution : une ligne JSONL par appel LLM, ecrite au moment de l'appel.

    Modes : "inactif" (aucune trace), "enregistrer" (tout appel part au modele et s'inscrit),
    "rejouer" (tout vient du journal, une cle absente arrete la mesure), "mixte" (le journal
    sert ce qu'il connait, une cle absente part au modele et s'inscrit).
    La cle est l'empreinte du contenu exact envoye, indexee par occurrence.
    """

    def __init__(self):
        self.mode = "inactif"
        self.index = {}            # cle -> lignes enregistrees, dans l'ordre des occurrences
        self.vus = Counter()       # occurrences deja consommees, par cle
        self.stats = Counter()     # (role, "journal" | "appel")
        self.lignes = []           # lignes inscrites pendant la session courante
        self.notes = {}            # (cle, occurrence) -> champs parses lors d'un rejeu
        self.fichier = None
        self._derniere = None
        self._ecrites = 0

    @staticmethod
    def cle(role, contenu, graine, max_tokens):
        brut = json.dumps([MODELE, role, contenu, graine, max_tokens], ensure_ascii=False)
        return hashlib.sha256(brut.encode("utf-8")).hexdigest()[:20]

    def ouvrir(self, mode, fichier=None, source=()):
        self.fermer()
        self.mode, self.fichier = mode, fichier
        self.index, self.vus, self.stats = {}, Counter(), Counter()
        self.lignes, self.notes, self._derniere, self._ecrites = [], {}, None, 0
        for ligne in source:
            self.index.setdefault(ligne["cle"], []).append(ligne)
        if fichier is not None:
            fichier.parent.mkdir(parents=True, exist_ok=True)
            fichier.write_text("", encoding="utf-8")

    def consulter(self, role, contenu, graine, max_tokens=None):
        # Rend la reponse enregistree, ou None si le contenu n'a pas encore ete vu.
        if self.mode not in ("rejouer", "mixte"):
            return None
        k = self.cle(role, contenu, graine, max_tokens)
        n = self.vus[k]
        stock = self.index.get(k, [])
        if n < len(stock):
            self.vus[k] += 1
            self.stats[(role, "journal")] += 1
            self._derniere = ("journal", k, n)
            return stock[n]["reponse"]
        if self.mode == "rejouer":
            raise RuntimeError("rejeu interrompu : cle absente du journal (role %s)" % role)
        return None

    def inscrire(self, role, contenu, graine, reponse, max_tokens=None):
        self._vider()
        k = self.cle(role, contenu, graine, max_tokens)
        n = self.vus[k]
        self.vus[k] += 1
        self.stats[(role, "appel")] += 1
        if self.mode in ("enregistrer", "mixte"):
            self.lignes.append({"role": role, "cle": k, "occurrence": n, "graine": graine,
                                "max_tokens": max_tokens, "contenu": contenu,
                                "reponse": reponse})
            self._derniere = ("appel", k, n)

    def noter(self, **champs):
        # Champs parses par l'appelant, rattaches a la derniere reponse servie ou inscrite.
        if self._derniere is None:
            return
        source, k, n = self._derniere
        if source == "appel":
            self.lignes[-1].update(champs)
        else:
            self.notes[(k, n)] = champs

    def _vider(self):
        # Ecrit sur disque les lignes completes : la trace suit l'execution.
        if self.fichier is None:
            return
        with self.fichier.open("a", encoding="utf-8") as f:
            for ligne in self.lignes[self._ecrites:]:
                f.write(json.dumps(ligne, ensure_ascii=False) + "\n")
        self._ecrites = len(self.lignes)

    def fermer(self):
        if self.mode in ("enregistrer", "mixte"):
            self._vider()
        self._derniere = None

    def appels(self):
        return sum(v for (r, s), v in self.stats.items() if s == "appel")

    def servis(self):
        return sum(v for (r, s), v in self.stats.items() if s == "journal")


JOURNAL = Journal()
DOSSIER_JOURNAL = Path(tempfile.gettempdir()) / "coursia-qc-py-44"   # hors depot


def appel(systeme, utilisateur, graine, max_tokens=MAX_TOKENS, role="?"):
    # Un appel au modele, avec reprises bornees, servi par le journal quand il le connait.
    contenu = [systeme, utilisateur]
    rep = JOURNAL.consulter(role, contenu, graine, max_tokens)
    if rep is not None:
        return rep
    derniere = None
    for essai in range(4):
        try:
            rep = client.chat.completions.create(
                model=MODELE,
                messages=[{"role": "system", "content": systeme},
                          {"role": "user", "content": utilisateur}],
                max_completion_tokens=max_tokens,
                seed=graine * 1000 + essai,
            )
            APPELS["n"] += 1
            texte = rep.choices[0].message.content or ""
            JOURNAL.inscrire(role, contenu, graine, texte, max_tokens)
            return texte
        except Exception as e:                      # noqa: BLE001 - on veut le message
            derniere = e
            time.sleep(2.0 * (essai + 1))
    raise RuntimeError("appel LLM defaillant apres 4 essais : %s" % derniere)


def extraire_json(txt):
    # Tolere les clôtures markdown et le bavardage autour de l'objet.
    m = re.search(r"\{.*\}", txt or "", re.S)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
    except Exception:
        return None
    return obj if isinstance(obj, dict) else None


def _nombre(x, defaut=0.0):
    try:
        v = float(x)
        return v if math.isfinite(v) else defaut
    except Exception:
        return defaut


def _texte_ctx(c):
    return ("Actif : %s | date : %s | cloture : %.2f\n"
            "Rendement cumule 20 seances : %+.2f%%\n"
            "Volatilite annualisee 20 seances : %.2f%%\n"
            "Drawdown vs plus haut 60 seances : %+.2f%%\n"
            "Z-score du prix vs moyenne 20 seances : %+.2f\n"
            "Asymetrie des 60 derniers rendements : %+.2f\n"
            "Rendements des 5 dernieres seances (%% ) : %s"
            % (ACTIF, c["date"], c["cloture"], c["ret20"] * 100, c["vol20"] * 100,
               c["drawdown60"] * 100, c["zscore20"], c["skew60"],
               ", ".join("%+.2f" % x for x in c["cinq_jours"])))


SCHEMA = ('Reponds STRICTEMENT par un objet JSON, sans texte autour : '
          '{"biais": "haussier"|"baissier"|"neutre", "prevision_pct": <nombre>, '
          '"argument": "<deux phrases maximum, avec les chiffres du contexte>"}. '
          '"prevision_pct" est le rendement attendu en pourcentage sur les 5 prochaines seances.')

SYST_MOMENTUM = ("Tu es analyste momentum sur un desk crypto. Tu juges la poursuite ou "
                 "l'epuisement d'une tendance a partir des rendements, du drawdown et de la "
                 "position par rapport a la moyenne. " + SCHEMA)
SYST_RISQUE = ("Tu es analyste risque sur un desk crypto. Tu juges la compression ou "
               "l'expansion de la volatilite, la asymetrie des rendements et le retour a la "
               "moyenne apres un mouvement extreme. " + SCHEMA)
SYST_PM = ('Tu es le gestionnaire de portefeuille. Tu tranches apres avoir lu deux analyses '
           'et un debat. Reponds STRICTEMENT par un objet JSON : '
           '{"decision": "LONG"|"FLAT"|"SHORT", "prevision_pct": <nombre>, '
           '"confiance": <nombre entre 0 et 1>, "justification": "<deux phrases>"}.')


@dataclass
class Reponse:
    biais: str = "neutre"          # pour les analystes
    decision: str = "FLAT"         # pour le PM et la baseline
    prevision: float = 0.0
    texte: str = ""
    brut: str = ""
    parse: bool = True


def analyste(systeme, c, graine):
    role = "momentum" if systeme is SYST_MOMENTUM else "risque"
    brut = appel(systeme, _texte_ctx(c), graine, role=role)
    obj = extraire_json(brut)
    if obj is None or "prevision_pct" not in obj:
        JOURNAL.noter(parse=False, decision=None, prevision_pct=None)
        return Reponse(texte=brut[:200], brut=brut, parse=False)
    r = Reponse(biais=str(obj.get("biais", "neutre")).lower(),
                prevision=_nombre(obj.get("prevision_pct")),
                texte=str(obj.get("argument", ""))[:400], brut=brut)
    JOURNAL.noter(parse=True, decision=r.biais, prevision_pct=r.prevision)
    return r


print("client pret | appels :", APPELS["n"], "| journal :", JOURNAL.mode)

client pret | appels : 0 | journal : inactif


### Lecture — un câblage vérifié à coût nul

`client pret | appels : 0`. La cellule instancie le client LLM, vérifie sa configuration et n'envoie **rien** : le compteur d'appels, qui restera l'unité de coût du carnet (295 au total en section 7), est à zéro.

C'est le patron « câblage d'abord, consommation ensuite » : une erreur d'URL, de modèle ou d'authentification serait attrapée ici, avant d'être payée 295 fois. Le compteur mesure le **protocole** — combien d'appels la logique du carnet déclenche, au point de décision comme dans le débat, indépendamment du volume échangé : c'est ce qui permet de chiffrer le coût exactement, décision par décision, avant de relancer la campagne.

## 4. Le débat bull/bear — organe `AgentGroupChat`

Le débat est le seul endroit où les agents se parlent. Il est borné à **deux tours**
(Bull, Bear, Bull, Bear) : au-delà, les positions se répètent et le coût croît sans
information. Chaque intervenant reçoit le contexte chiffré **et** les deux analyses, et doit
chiffrer sa position — un débat sans nombre ne se solde pas.

In [6]:
from semantic_kernel.agents import AgentGroupChat, ChatCompletionAgent
from semantic_kernel.agents.strategies import SequentialSelectionStrategy
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import AuthorRole, ChatMessageContent

SERVICE = OpenAIChatCompletion(ai_model_id=MODELE)   # lit OPENAI_API_KEY

CONSIGNES = {
    "Bull": ("Tu defends le scenario haussier. Deux phrases maximum. Tu dois citer "
             "un chiffre du contexte et avancer une prevision chiffree a 5 seances."),
    "Bear": ("Tu defends le scenario baissier. Deux phrases maximum. Tu dois citer "
             "un chiffre du contexte et avancer une prevision chiffree a 5 seances."),
}


async def debattre(c, a_momentum, a_risque, tours=2, graine=None, consignes=None):
    # Deux agents, selection sequentielle, arret dur apres 2*tours messages.
    # Le journal sert le prefixe du debat qu'il connait ; la suite part au modele,
    # dans un AgentGroupChat amorce par les messages deja servis.
    consigne = _texte_ctx(c) + (
        "\n\nAnalyse momentum : biais %s, prevision %+.2f%%, argument : %s"
        "\nAnalyse risque   : biais %s, prevision %+.2f%%, argument : %s"
        % (a_momentum.biais, a_momentum.prevision, a_momentum.texte,
           a_risque.biais, a_risque.prevision, a_risque.texte))
    instr = dict(CONSIGNES, **(consignes or {}))
    ordre = ["Bull", "Bear"] * tours

    messages = []
    while len(messages) < len(ordre):
        nom = ordre[len(messages)]
        rep = JOURNAL.consulter("debat:" + nom, [instr[nom], consigne, messages], graine)
        if rep is None:
            break
        messages.append((nom, rep))
    if len(messages) == len(ordre):
        return messages

    bull = ChatCompletionAgent(service=SERVICE, name="Bull", instructions=instr["Bull"])
    bear = ChatCompletionAgent(service=SERVICE, name="Bear", instructions=instr["Bear"])
    # La selection sequentielle part du PREMIER agent de la liste, quel que soit
    # l'historique deja present : la liste est ordonnee pour que l'orateur attendu
    # ouvre la reprise (sinon, apres un Bull servi, Bull reparlerait).
    agents = [bull, bear] if ordre[len(messages)] == "Bull" else [bear, bull]
    chat = AgentGroupChat(agents=agents, selection_strategy=SequentialSelectionStrategy())
    await chat.add_chat_message(ChatMessageContent(role=AuthorRole.USER, content=consigne))
    for nom, texte in messages:
        await chat.add_chat_message(
            ChatMessageContent(role=AuthorRole.ASSISTANT, content=texte, name=nom))

    async for m in chat.invoke():
        nom, texte = (m.name or "?"), (m.content or "").strip()
        APPELS["n"] += 1
        JOURNAL.inscrire("debat:" + nom, [instr.get(nom, ""), consigne, list(messages)],
                         graine, texte)
        messages.append((nom, texte))
        if len(messages) >= len(ordre):
            break
    # Garde : l'alternance Bull/Bear doit etre exactement celle du protocole.
    assert [n for n, _ in messages] == ordre, [n for n, _ in messages]
    return messages


# Demonstration sur le premier point de decision, avant la campagne complete.
demo = await debattre(CTX[0], analyste(SYST_MOMENTUM, CTX[0], 0), analyste(SYST_RISQUE, CTX[0], 0))
for nom, texte in demo:
    print("[%s] %s" % (nom, texte.replace("\n", " ")[:230]))
print("\nmessages de debat :", len(demo), "| appels cumules :", APPELS["n"])

[Bull] Le scénario haussier reste privilégié, porté par le rendement cumulé de **+6,34 % sur 20 séances** et le drawdown limité à **-2,15 %**.   Prévision à 5 séances : **+1,80 %**, soit environ **44 781 USD**.
[Bear] Le scénario baissier reste à privilégier, car la volatilité annualisée élevée de **42,70 %** et les récents mouvements heurtés signalent un risque de retournement malgré le gain de 6,34 % sur 20 séances.   Prévision à 5 séances : 
[Bull] Le scénario haussier reste privilégié, soutenu par le rendement cumulé de **+6,34 % sur 20 séances** et le drawdown limité à **-2,15 %**.   Prévision à 5 séances : **+1,80 %**, soit environ **44 781 USD**.
[Bear] Le scénario baissier reste privilégié, car la volatilité annualisée de **42,70 %** et les rendements récents heurtés augmentent le risque de retournement.   Prévision à 5 séances : **-2,00 %**, soit environ **43 109 USD**.

messages de debat : 4 | appels cumules : 6


### Lecture — le débat : deux lectures symétriques du même contexte

La cellule joue un tour de débat complet sur un contexte réel. Les deux rôles reçoivent le même marché (`+6,34 % sur 20 séances`) et divergent exactement sur ce qu'ils en font :

- **Le Bull projette** : `+1,80 % vers 44 781 $`. Il extrapole la tendance courte — la cible chiffrée donne l'apparence d'une prévision quantitative, mais c'est une lecture *directionnelle* : la poursuite du mouvement observé.
- **Le Bear refuse la projection** : `vol annualisée 42,70 %`. Il ne donne pas de cible — il répond à une échelle différente : le risque. À 42,7 % de volatilité annualisée, un mouvement de +1,80 % sur 5 séances est bien dans le bruit quotidien de l'actif ; le Bear ne dit pas « ça va baisser », il dit « votre question n'a pas de réponse à cette précision ».

C'est tout l'intérêt du dispositif d'information asymétrique : chaque analyste est une **hypothèse sur la fenêtre d'information qui suffit à décider** (le Bull regarde la dernière valeur, le Bear l'historique, l'analyste de flux de l'exercice 1 ne verra que 5 rendements). Le cadre arbitre entre lectures incommensurables — c'est un rôle plus difficile que de « choisir le bon analyste », et la section 8 en mesure l'enjeu : les deux analystes ne sont d'accord entre eux que dans 40,6 % des cas (13/32). L'arbitrage de lectures divergentes est la règle du dispositif, pas l'exception.

À noter pour la suite : ce tour de démonstration consomme des appels (il entrera dans les 7 de différence entre 288 et 295 au compteur final) — le débat a un prix, mesuré dès la section suivante.

## 5. La mémoire de décisions soldées

C'est la pièce que le cadre ajoute à un débat : une décision n'est pas seulement prise, elle
est **soldée**. À l'échéance de détention, l'écart entre la prévision et le rendement réalisé
devient une leçon d'une phrase, et cette leçon revient dans le prompt du gérant.

Deux propriétés rendent la mémoire mesurable :

- **Au premier point de décision, elle est vide** — c'est le témoin négatif structurel : si le
  cadre gagne, ce n'est pas grâce à elle, puisqu'elle ne peut pas encore exister.
- **Chaque entrée est soldée avant d'être lue.** Les points sont espacés de 120 jours et
  l'horizon est de 5 séances : aucune entrée non soldée ne peut entrer dans le prompt. Le
  carnet vérifie cette propriété et lève si elle est violée.

In [7]:
class Memoire:
    # Journal de decisions soldees. Rien n'entre ici tant que le rendement est inconnu.
    def __init__(self):
        self.entrees = []

    def ajouter(self, date, decision, prevision, realise, reflexion):
        self.entrees.append({"date": date, "decision": decision,
                             "prevision": prevision, "realise": realise,
                             "reflexion": reflexion})

    def toutes_soldees(self):
        return all(e["realise"] is not None for e in self.entrees)

    def resume(self, k=3):
        soldees = [e for e in self.entrees if e["realise"] is not None]
        if not soldees:
            return "(memoire vide : aucune decision soldee a ce stade)"
        lignes = []
        for e in soldees[-k:]:
            lignes.append("- %s : %s, prevu %+.2f%%, realise %+.2f%% -> %s"
                          % (e["date"], e["decision"], e["prevision"],
                             e["realise"] * 100, e["reflexion"]))
        return "\n".join(lignes)

    def __len__(self):
        return len(self.entrees)


def pm(c, a_momentum, a_risque, debat, memoire, graine):
    fil = "\n".join("[%s] %s" % (n, t) for n, t in debat)
    demand = (_texte_ctx(c)
              + "\n\nAnalyse momentum : %s (%+.2f%%)\nAnalyse risque : %s (%+.2f%%)"
              % (a_momentum.biais, a_momentum.prevision, a_risque.biais, a_risque.prevision)
              + "\n\nDebat :\n" + fil
              + "\n\nMemoire des decisions soldees (les plus recentes) :\n" + memoire.resume(3)
              + "\n\nTranche. Rappelle-toi que la memoire ne contient que des decisions "
                "deja soldees contre le rendement realise.")
    brut = appel(SYST_PM, demand, graine, role="pm")
    obj = extraire_json(brut)
    if obj is None or "decision" not in obj:
        JOURNAL.noter(parse=False, decision="FLAT", prevision_pct=0.0)
        return Reponse(brut=brut, parse=False)
    dec = str(obj.get("decision", "FLAT")).upper()
    dec = dec if dec in ("LONG", "FLAT", "SHORT") else "FLAT"
    r = Reponse(decision=dec, prevision=_nombre(obj.get("prevision_pct")),
                texte=str(obj.get("justification", ""))[:400], brut=brut)
    JOURNAL.noter(parse=True, decision=r.decision, prevision_pct=r.prevision)
    return r


SYST_REFLEXION = ("Tu tires une lecon d'une decision soldee. Reponds par UNE phrase, "
                  "reutilisable pour la prochaine decision, sans compliment ni excuse.")


def reflexion(decision, prevision, realise, graine):
    q = ("Decision : %s. Prevision : %+.2f%%. Rendement realise sur la periode de detention : "
         "%+.2f%%. Quelle lecon d'une phrase ?" % (decision, prevision, realise * 100))
    return appel(SYST_REFLEXION, q, graine, max_tokens=90,
                 role="reflexion").strip().replace("\n", " ")[:240]


memoire_test = Memoire()
print("memoire vide ->", len(memoire_test), "entree(s)")
print(memoire_test.resume())

memoire vide -> 0 entree(s)
(memoire vide : aucune decision soldee a ce stade)


### Lecture — l'état initial de la mémoire

`memoire vide -> 0 entree(s)` : au premier point de décision de la campagne, l'agent ne sait rien de ses performances passées — ni ses gains, ni ses biais, ni si ses prévisions précédentes étaient justes. Cette ligne est la ligne de référence de tout ce qui suit en section 8 :

- **La mémoire se remplira de 0 à 7 entrées** (`[0..7]` mesuré en section 8) : une par point de décision, après le soldage. Vérifier ce remplissage est une sonde d'intégrité — un trou dans la numérotation signalerait une décision non soldée (et donc un horizon interrompu), un doublon une décision comptée deux fois.
- **Le premier arbitrage se fait donc à froid.** Le cadre décide au point 0 sans retour d'expérience, exactement comme un arbitre humain débutant : son seul bagage est le prompt système et le contexte de marché. La différence entre le premier et le dernier point de décision (mémoire de 7 entrées) est empiriquement mesurable dans le carnet — et c'est la porte d'entrée de l'exercice 2 : si la mémoire aide, aide-t-elle parce qu'elle est récente ou parce qu'elle est *pertinente* ?
- **0 entrée est aussi un choix de conception.** Une mémoire amorcée à la main (exemples pré-écrits) rendrait le premier point de décision non reproductible ; partir vide garantit que toute l'expérience accumulée provient des décisions réelles du run — la traçabilité complète, au prix d'un arbitrage initial naïf.

## 6. Le bras baseline

Le témoin est le même modèle, le même contexte chiffré, **un seul appel** : ni analyste, ni
débat, ni mémoire. C'est le concurrent honnête — si le cadre multi-agents ne le bat pas, c'est
que le protocole n'apporte rien que le modèle ne savait déjà faire seul.

In [8]:
SYST_BASELINE = ('Tu es analyste de marche. A partir du contexte, prevois le rendement des 5 '
                 'prochaines seances. Reponds STRICTEMENT par un objet JSON : '
                 '{"prevision_pct": <nombre>, "decision": "LONG"|"FLAT"|"SHORT", '
                 '"argument": "<une phrase>"}.')


def baseline(c, graine):
    brut = appel(SYST_BASELINE, _texte_ctx(c), graine, role="baseline")
    obj = extraire_json(brut)
    if obj is None or "prevision_pct" not in obj:
        JOURNAL.noter(parse=False, decision="FLAT", prevision_pct=0.0)
        return Reponse(brut=brut, parse=False)
    dec = str(obj.get("decision", "FLAT")).upper()
    dec = dec if dec in ("LONG", "FLAT", "SHORT") else "FLAT"
    r = Reponse(decision=dec, prevision=_nombre(obj.get("prevision_pct")),
                texte=str(obj.get("argument", ""))[:300], brut=brut)
    JOURNAL.noter(parse=True, decision=r.decision, prevision_pct=r.prevision)
    return r


avant = APPELS["n"]
essai_bl = baseline(CTX[0], 0)
print("baseline sur %s : decision %s, prevision %+.2f%% (%s)"
      % (CTX[0]["date"], essai_bl.decision, essai_bl.prevision,
         "parse" if essai_bl.parse else "NON PARSE"))
print("appels consommes par le bras baseline :", APPELS["n"] - avant)

baseline sur 2024-01-06 : decision LONG, prevision +1.20% (parse)
appels consommes par le bras baseline : 1


### Lecture — la baseline, un témoin de protocole réduit à un appel

`baseline sur 2024-01-06 : decision LONG, prevision +1.20% (parse), appels 1`. Trois choses se lisent sur cette ligne unique :

1. **La baseline décide seule, sans le dispositif.** Elle demande au LLM, en un appel unique, une prévision **et** une décision (`+1.20%`, `LONG`, `(parse)` — le format a été respecté) — le même modèle sur le même contexte, mais sans débat, sans mémoire et sans arbitre. Elle prend position dans les trois valeurs du vocabulaire : la campagne de la section 8 en comptera 14 LONG / 9 FLAT / 9 SHORT pour ce bras. Ce n'est pas un témoin muet, c'est un décideur **réduit à sa plus simple expression** — et la différence cadre-baseline mesurée plus loin porte sur le protocole entier (débat, mémoire, arbitrage), pas sur la seule qualité de l'information reçue.
2. **Son coût unitaire est 1 appel** — contre ~9 pour une décision complète du cadre. Le témoin est ~9 fois moins cher que le traité : c'est ce qui rend légitime de le faire tourner sur les 32 paires sans doubler le budget.
3. **`(parse)` est une petite ligne qui porte beaucoup.** La suite de la campagne comptera `non parsees: baseline 0` sur l'ensemble des appels : la baseline ne rate jamais son format. Le pipeline le plus simple est aussi le plus fiable — chaque rôle ajouté au débat (analystes) apportera de l'information **et** des échecs de parsing (14/64). La robustesse décroît avec la richesse du dispositif : c'est un arbitrage, pas un accident.

## 7. La campagne

Chaque graine parcourt les huit points de décision **dans l'ordre chronologique** : c'est la
seule façon dont la mémoire peut se remplir et être lue. Pour chaque point, les deux bras
tournent sur le **même contexte**, ce qui rend les pertes appariées.

Coût : à chaque point et par graine, 2 analyses + 4 messages de débat + 1 décision de PM +
1 réflexion + 1 baseline = **9 appels**. Soit `8 x 4 x 9 = 288` appels **pour la campagne**. S'y ajoutent les **7 appels des
cellules de démonstration** plus haut — 2 analyses, les 4 messages du débat de
démonstration (la cellule du débat imprime `appels cumules : 6`) et la baseline de
démonstration — d'où les **295** que la cellule suivante imprime au compteur. C'est de
loin la plus longue cellule du carnet.

In [9]:
def perte(prevision_pct, realise):
    # Perte de prevision : erreur absolue en points de pourcentage, comparable entre bras.
    return abs(prevision_pct / 100.0 - realise)


async def campagne(graines, contextes, consignes=None, trace=True):
    # Une graine = une memoire neuve qui parcourt les points dans l'ordre chronologique.
    lignes = []
    for graine in graines:
        memoire = Memoire()
        for i, c in enumerate(contextes):
            a_mom = analyste(SYST_MOMENTUM, c, graine)
            a_ris = analyste(SYST_RISQUE, c, graine)
            debat = await debattre(c, a_mom, a_ris, graine=graine, consignes=consignes)
            decision = pm(c, a_mom, a_ris, debat, memoire, graine)

            # La memoire lue par le PM ne doit contenir que des decisions soldees.
            assert memoire.toutes_soldees(), "memoire non soldee lue par le PM"

            base = baseline(c, graine)
            realise = c["realise_5j"]

            lecon = reflexion(decision.decision, decision.prevision, realise, graine)
            memoire.ajouter(c["date"], decision.decision, decision.prevision, realise, lecon)

            lignes.append({
                "graine": graine, "date": c["date"], "realise": realise,
                "prev_pm": decision.prevision, "dec_pm": decision.decision,
                "prev_bl": base.prevision, "dec_bl": base.decision,
                "parse_pm": decision.parse, "parse_bl": base.parse,
                "biais_mom": a_mom.biais, "biais_ris": a_ris.biais,
                "memoire_avant": i, "parse_analystes": int(a_mom.parse) + int(a_ris.parse),
            })
        if trace:
            print("graine %-3d terminee | appels cumules %d" % (graine, APPELS["n"]))
    return pd.DataFrame(lignes)


APPELS_PAR_POINT = 9      # 2 analystes, 4 messages de debat, PM, baseline, reflexion
annonce = len(SEEDS) * len(CTX) * APPELS_PAR_POINT
print("appels annonces pour la campagne : %d graines x %d points x %d = %d"
      % (len(SEEDS), len(CTX), APPELS_PAR_POINT, annonce))

FICHIER_JOURNAL = DOSSIER_JOURNAL / "journal_campagne.jsonl"
avant_campagne = APPELS["n"]
debut = time.time()
JOURNAL.ouvrir("enregistrer", FICHIER_JOURNAL)
res = await campagne(SEEDS, CTX)
JOURNAL.fermer()
JOURNAL_CAMPAGNE = list(JOURNAL.lignes)

res["perte_pm"] = [perte(p, r) for p, r in zip(res["prev_pm"], res["realise"])]
res["perte_bl"] = [perte(p, r) for p, r in zip(res["prev_bl"], res["realise"])]
res["d"] = res["perte_pm"] - res["perte_bl"]      # negatif = le cadre fait mieux

print("\nduree      : %.1f min | appels LLM cumules : %d" % ((time.time() - debut) / 60.0, APPELS["n"]))
print("appels de la campagne : annonces %d | releves %d"
      % (annonce, APPELS["n"] - avant_campagne))
print("journal    : %s, %d lignes, %.0f Ko (hors depot)"
      % (FICHIER_JOURNAL.name, len(JOURNAL_CAMPAGNE), FICHIER_JOURNAL.stat().st_size / 1024))
print("paires     :", len(res))
print("non parsees: PM %d, baseline %d, analystes %d/%d"
      % ((~res["parse_pm"]).sum(), (~res["parse_bl"]).sum(),
         len(res) * 2 - res["parse_analystes"].sum(), len(res) * 2))
res.head(8)

appels annonces pour la campagne : 4 graines x 8 points x 9 = 288


graine 0   terminee | appels cumules 79


graine 1   terminee | appels cumules 151


graine 7   terminee | appels cumules 223


graine 42  terminee | appels cumules 295

duree      : 11.4 min | appels LLM cumules : 295
appels de la campagne : annonces 288 | releves 288
journal    : journal_campagne.jsonl, 288 lignes, 487 Ko (hors depot)
paires     : 32
non parsees: PM 0, baseline 0, analystes 6/64


,graine,date,realise,prev_pm,dec_pm,prev_bl,dec_bl,parse_pm,parse_bl,biais_mom,biais_ris,memoire_avant,parse_analystes,perte_pm,perte_bl,d
0,0,2024-01-06,0.054090,0.8,LONG,1.2,LONG,True,True,haussier,neutre,0,2,0.046090,0.042090,0.004000
1,0,2024-05-05,-0.050575,0.5,FLAT,0.8,LONG,True,True,neutre,neutre,1,2,0.055575,0.058575,-0.003000
2,0,2024-09-02,-0.084124,0.0,FLAT,-0.8,FLAT,True,True,neutre,haussier,2,2,0.084124,0.076124,0.008000
3,0,2024-12-31,0.052294,-0.5,FLAT,-1.2,SHORT,True,True,baissier,neutre,3,1,0.057294,0.064294,-0.007000
4,0,2025-04-30,0.005740,1.5,LONG,1.2,LONG,True,True,haussier,haussier,4,2,0.009260,0.006260,0.003000
5,0,2025-08-28,-0.011944,-0.8,SHORT,-1.2,SHORT,True,True,baissier,neutre,5,2,0.003944,0.000056,0.003888
6,0,2025-12-26,0.002376,0.0,FLAT,-1.2,SHORT,True,True,baissier,neutre,6,2,0.002376,0.014376,-0.012000
7,0,2026-04-25,-0.016849,1.7,LONG,1.8,LONG,True,True,haussier,haussier,7,2,0.033849,0.034849,-0.001000


### Lecture — le coût de la campagne, lu au compteur

La sortie donne la trajectoire du coût : **79 → 151 → 223 → 295 appels cumulés**, pour **32 paires** de décision/rendement, en **11,4 min**.

- **La cohérence arithmétique se vérifie en croisant les sections.** 4 graines × 8 points de décision = 32 paires — exactement le compteur. Les incréments de 72 appels par graine (295 − 223, etc.) divisés par 8 points donnent **9 appels par décision** : le cadre, les analystes et les tours de débat se paient au même tarif à chaque point. La ligne `appels de la campagne : annonces 288 | releves 288` fait le même contrôle de l'intérieur : le nombre posé d'avance et le nombre relevé dans le journal coïncident.
- **La robustesse du parsing est inégale selon le rôle.** `non parsees: PM 0, baseline 0, analystes 6/64` : les prompts structurés (portfolio manager, baseline) passent sans perte, les analystes laissent passer 6 réponses hors format sur 64 — **9,4 %**. Ce compte varie d'une exécution à l'autre du même code (de 6 à 14 sur 64 dans nos essais) : la fragilité tient à la forme libre des réponses du débat, son ampleur au tirage. Le débat est donc à la fois le mécanisme le plus coûteux (la majorité des 9 appels) et le plus fragile du pipeline. C'est un résultat en soi : fiabiliser les réponses des analystes vaudrait plus que d'en ajouter un (exercice 1).
- **Le coût marginal d'une expérience.** À ~9 appels par décision et ~2,9 min par graine (11,4 / 4), doubler les graines ou les points double la facture en appels comme en temps : ces compteurs sont ce qui transforme « il suffirait de plus de données » en décision chiffrée. La section 9 y revient — 288 appels pour la campagne, 295 au compteur avec les 7 appels de démonstration des sections précédentes — et la section 10 montre comment le journal ramène une ré-exécution à zéro appel.

## 8. Le verdict — doctrine §C

Trois conditions, **toutes** requises pour `BEATS` :

1. l'écart de perte `edge = moyenne(perte_baseline) - moyenne(perte_cadre)` est **au moins
   deux écarts-types** de sa dispersion entre graines ;
2. le Diebold-Mariano sur les pertes appariées est significatif (`p < 0,05`) ;
3. sa statistique est **négative** — c'est-à-dire que c'est bien le cadre qui a la perte la
   plus faible.

À quoi s'ajoute la lecture économique : la position implicite (`LONG`/`SHORT`/`FLAT`) est
tenue sur l'horizon, nette de **10 points de base par côté**.

In [10]:
def diebold_mariano(d, h=1):
    # Newey-West HAC, lag h. d = perte_modele - perte_baseline ; stat < 0 => modele meilleur.
    d = np.asarray(d, dtype=float)
    n = len(d)
    m = d.mean()
    gamma0 = np.sum((d - m) ** 2) / n
    s = gamma0
    for lag in range(1, h + 1):
        g = np.sum((d[lag:] - m) * (d[:-lag] - m)) / n
        s += 2.0 * (1.0 - lag / (h + 1.0)) * g
    stat = m / math.sqrt(s / n) if s > 0 else 0.0
    p = math.erfc(abs(stat) / math.sqrt(2.0))
    return float(m), float(stat), float(p)


par_graine = res.groupby("graine").apply(
    lambda g: pd.Series({"perte_pm": g["perte_pm"].mean(),
                         "perte_bl": g["perte_bl"].mean()}))
par_graine["edge"] = par_graine["perte_bl"] - par_graine["perte_pm"]

edge = float(par_graine["edge"].mean())
sigma = float(par_graine["edge"].std(ddof=1))
ratio = edge / sigma if sigma > 0 else float("inf")
moy_d, stat, pval = diebold_mariano(res["d"])

print(par_graine.round(5).to_string())
print("\nedge (points de %%)      : %+.4f" % (edge * 100))
print("sigma inter-graines     : %.4f  (n=%d)" % (sigma * 100, len(SEEDS)))
print("edge / sigma            : %+.2f  (seuil §C : >= 2)" % ratio)
print("Diebold-Mariano         : stat %+.3f, p = %.4f (seuil : p < 0,05 et stat < 0)"
      % (stat, pval))

conjonction = (ratio >= 2.0) and (pval < 0.05) and (stat < 0)
if conjonction:
    verdict = "BEATS"
elif ratio <= -2.0 and pval < 0.05 and stat > 0:
    verdict = "NO BEATS"
else:
    verdict = "INCONCLUSIVE"
print("\nVERDICT :", verdict)

        perte_pm  perte_bl     edge
graine                             
0        0.03656   0.03708  0.00051
1        0.03558   0.03658  0.00100
7        0.03856   0.03635 -0.00222
42       0.03819   0.03550 -0.00269

edge (points de %)      : -0.0848
sigma inter-graines     : 0.1874  (n=4)
edge / sigma            : -0.45  (seuil §C : >= 2)
Diebold-Mariano         : stat +0.758, p = 0.4485 (seuil : p < 0,05 et stat < 0)

VERDICT : INCONCLUSIVE


### Lecture — les trois nombres du verdict

La sortie porte quatre edge par graine (`0.00051 / 0.00100 / -0.00222 / -0.00269`) puis trois nombres qui décident de tout :

1. **`edge -0.0848, sigma 0.1874 (n=4)` → `edge/sigma -0.45` contre un seuil `>= 2`.** L'edge moyen est négatif — en moyenne, la prévision du cadre perd un peu plus que celle de la baseline — et il vaut moins d'un demi-sigma de dispersion inter-graines. Deux graines sont positives, deux négatives, avec des écarts du même ordre : l'effet ne domine pas sa propre variance, et son signe n'est pas même établi.
2. **`Diebold-Mariano stat +0.758, p = 0.4485`.** Le test compare les prévisions du cadre à celles de la baseline paire par paire : p = 0,45, loin du seuil 0,05, et la statistique est positive, du mauvais côté du critère (`stat < 0`). À 32 paires, aucune différence de précision n'est établie.
3. **`VERDICT : INCONCLUSIVE`** — le verdict honnête à cette taille d'expérience. Le protocole pré-annoncé (seuil `>= 2` ET DM < 0,05 ET comparaison à la baseline) a fait son travail : refuser de trancher est une **réponse**, pas un échec du carnet.

Ce qu'il ne faut pas lire : « le cadre prévoit moins bien ». Ce que la sortie dit : « à 4 graines et 8 points de décision, l'écart de précision entre le cadre et la baseline n'est pas distinguable du bruit ». Le tableau qui suit donne un classement sur le Sharpe des **décisions** ; ce verdict porte sur la précision des **prévisions chiffrées**, et il en dit la fiabilité (non établie). Les deux se lisent ensemble : un classement sans test de significativité est une anecdote ; un test sans le classement est un aveugle.

Ce verdict n'est pas non plus stable d'une exécution à l'autre : le même code, relancé, rend des ratios `edge/sigma` d'ampleur et même de signe différents. La section 10 en donne la mesure et montre comment figer une exécution pour la rejouer à l'identique ; la section 9 creuse d'abord ce que ce verdict engage sur le cadrage de l'expérience.

In [11]:
# Lecture economique : la position implicite de chaque bras, nette de frais.
def pnl(decisions, realises, cout_bps=COST_BPS):
    signe = {"LONG": 1.0, "SHORT": -1.0, "FLAT": 0.0}
    p = []
    for dec, r in zip(decisions, realises):
        s = signe.get(dec, 0.0)
        frais = (2.0 * cout_bps / 10000.0) if s != 0.0 else 0.0
        p.append(s * r - frais)
    return np.asarray(p, dtype=float)


p_pm = pnl(res["dec_pm"], res["realise"])
p_bl = pnl(res["dec_bl"], res["realise"])
p_bh = pnl(["LONG"] * len(res), res["realise"])       # achat-conservation, meme cout

def sharpe_par_point(p):
    # Rendements par decision (et non journaliers) : on annualise par le nombre de
    # decisions par an, soit ~365/ESPACEMENT.
    m, s = p.mean(), p.std(ddof=1)
    if s == 0:
        return 0.0
    return float(m / s * np.sqrt(365.0 / ESPACEMENT))


print("Sharpe par decision (net de frais) :")
print("  cadre multi-agents : %+.3f" % sharpe_par_point(p_pm))
print("  baseline           : %+.3f" % sharpe_par_point(p_bl))
print("  achat-conservation : %+.3f" % sharpe_par_point(p_bh))
print("\nrendement moyen par decision : cadre %+.3f%% | baseline %+.3f%% | achat %+.3f%%"
      % (p_pm.mean() * 100, p_bl.mean() * 100, p_bh.mean() * 100))
print("frais retenus : %d bps par cote, aller-retour = %d bps" % (COST_BPS, 2 * COST_BPS))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(["cadre", "baseline", "achat"], [sharpe_par_point(p_pm), sharpe_par_point(p_bl),
                                        sharpe_par_point(p_bh)],
       color=["#2b6cb0", "#c05621", "#718096"])
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Sharpe par decision")
ax.set_title("QC-Py-44 - rendement risque net de frais, par bras")
plt.tight_layout()
plt.show()

Sharpe par decision (net de frais) :
  cadre multi-agents : -0.009
  baseline           : -0.224
  achat-conservation : -0.317

rendement moyen par decision : cadre -0.017% | baseline -0.455% | achat -0.812%
frais retenus : 10 bps par cote, aller-retour = 20 bps


### Lecture — le classement descriptif des trois bras

Le tableau croise trois stratégies jouées sur les mêmes 32 décisions :

- **cadre** : Sharpe `-0.009`, rendement moyen `-0.017%` par décision ;
- **baseline** (un appel, sans dispositif) : Sharpe `-0.224`, rendement `-0.455%` ;
- **achat-conservation** : Sharpe `-0.317`, rendement `-0.812%`.

Lecture en deux temps. **Le classement d'abord** : cadre > baseline > achat-conservation. Le bras LLM complet domine les deux témoins — mais les trois Sharpe sont négatifs et celui du cadre est à peine distinct de zéro : sur cette fenêtre, le marché n'offrait pas de rendement gratuit, et le classement se joue sur qui a le moins perdu. La distribution des décisions, imprimée plus bas avec les sondes, dit comment : le cadre s'abstient plus souvent que la baseline (13 FLAT contre 9) et vend moins à découvert (6 SHORT contre 10).

**La granularité des frais ensuite** : `frais 10 bps par cote, aller-retour = 20 bps`. Ces frais sont déjà retranchés par `pnl()` — un aller-retour complet par décision positionnée : le rendement moyen du cadre, -0,017 %, est donc **net de frais** (-1,7 bps nets par décision, aucune seconde déduction à opérer).

**Classement et verdict ne mesurent pas la même chose.** Le Sharpe juge les décisions (LONG, FLAT, SHORT) ; le verdict qui précède juge les prévisions chiffrées. Ici ils divergent : le cadre a le meilleur classement et une prévision très légèrement moins précise que la baseline. Rien d'incohérent — une prévision surestimée de près d'un point (voir les sondes) peut conduire à la même direction qu'une prévision juste. Ni l'un ni l'autre n'est établi à n=4 : l'écart de Sharpe est du même ordre que la dispersion des 32 décisions.

Le témoin le plus instructif est la **baseline** : le même modèle, réduit à un appel unique sans débat, sans mémoire et sans arbitre — son Sharpe de -0,224 dit ce que le protocole complet ajoute quand on le retire. La différence cadre-baseline (-0,009 contre -0,224) mesure l'effet du **protocole entier** (débat + mémoire + arbitrage, ~9 appels par décision contre 1) : elle ne sépare pas ces dimensions une à une — isoler le seul arbitre demanderait un témoin intermédiaire recevant le débat sans le cadre.

In [12]:
# Temoins et rapport de biais : ce que le cadre a reellement utilise.
# Tous les chiffres ci-dessous se calculent sur `res` : aucun appel LLM supplementaire.
accord = (res["biais_mom"] == res["biais_ris"]).mean()

print("=== Temoin : la memoire etait-elle vide au premier point ? ===")
print("  entrees en memoire au point 1, par construction : 0")
print("  entrees atteintes en fin de graine              :", N_POINTS - 1)
print("  colonne `memoire_avant` de la campagne          :", [int(x) for x in sorted(res["memoire_avant"].unique())])

print("\n=== Les deux analystes se contredisent-ils ? ===")
print("  taux d'accord momentum/risque : %.1f%% (%d/%d)"
      % (accord * 100, int(accord * len(res)), len(res)))

print("\n=== Biais signe moyen (prevision - realise), par bras ===")
for nom, col in (("cadre", "prev_pm"), ("baseline", "prev_bl")):
    b = (res[col] / 100.0 - res["realise"]).mean()
    print("  %-9s : %+.4f pt  (%s)" % (nom, b * 100,
                                        "surestime" if b > 0 else "sous-estime"))

print("\n=== Ecart-type des previsions (amplitude osee par chaque bras) ===")
print("  cadre    : %.3f pt" % (res["prev_pm"] / 100.0).std())
print("  baseline : %.3f pt" % (res["prev_bl"] / 100.0).std())

print("\n=== Distribution des decisions ===")
print(pd.DataFrame({"cadre": res["dec_pm"].value_counts(),
                    "baseline": res["dec_bl"].value_counts()}).fillna(0).astype(int))

=== Temoin : la memoire etait-elle vide au premier point ? ===
  entrees en memoire au point 1, par construction : 0
  entrees atteintes en fin de graine              : 7
  colonne `memoire_avant` de la campagne          : [0, 1, 2, 3, 4, 5, 6, 7]

=== Les deux analystes se contredisent-ils ? ===
  taux d'accord momentum/risque : 34.4% (11/32)

=== Biais signe moyen (prevision - realise), par bras ===
  cadre     : +0.9155 pt  (surestime)
  baseline  : +0.7608 pt  (surestime)

=== Ecart-type des previsions (amplitude osee par chaque bras) ===
  cadre    : 0.011 pt
  baseline : 0.011 pt

=== Distribution des decisions ===
       cadre  baseline
FLAT      13         9
LONG      13        13
SHORT      6        10


### Lecture — les trois sondes d'intégrité

Les sondes ne mesurent pas la performance (c'est fait plus haut) : elles vérifient que le dispositif a fonctionné comme conçu. Trois sondes, trois niveaux :

1. **La mémoire a bien traversé la campagne.** Les entrées vont de `[0..7]` : huit points de décision ont chacun laissé une trace, sans trou ni doublon. Partie de « memoire vide -> 0 entree(s) », elle s'est remplie exactement au rythme des décisions — c'est la condition pour que la question de l'exercice 2 (récence contre similarité) ait un objet à comparer.
2. **Les deux analystes ne s'accordent que dans 34,4 % des cas (11/32).** La sonde compare tour par tour le biais momentum et le biais risque — l'accord des **deux analystes entre eux**, pas celui du cadre avec le PM : dans 65,6 % des décisions, le cadre arbitre deux lectures **divergentes** du même contexte. C'est ce qu'on attend d'un dispositif d'information asymétrique : chaque analyste voit une fenêtre différente, et leur désaccord est la matière première de l'arbitrage. Si ce taux était proche de 100 %, le débat serait décoratif et un seul appel LLM suffirait.
3. **Le biais du cadre est de +0,9155 pt, en surestimation — et la baseline surestime aussi (+0,7608 pt).** Sur l'ensemble des décisions, la prévision dépasse le rendement réalisé de près d'un point : l'optimisme est d'abord celui du modèle, que le dispositif accentue d'environ 0,15 pt. C'est une information de premier ordre pour l'interprétation du verdict INCONCLUSIVE : l'edge mesuré plus haut (-0,0848 point de %) est plus de dix fois plus petit, en valeur absolue, que ce biais commun aux deux bras, ce qui borne ce qu'on peut conclure d'une campagne à 32 paires.

Ensemble, les trois sondes racontent la même chose : le verdict n'est pas une boîte noire, c'est une mesure dont on connaît les fuites (6/64 réponses d'analystes non parsées) et les biais (+0,9 pt). C'est ce qui distingue un protocole honnête d'un score ornemental.

## 9. Lecture du verdict

Trois choses se lisent ici, et il faut les séparer.

**Le verdict §C d'abord.** Il porte sur les **pertes de prévision** appariées, pas sur le
Sharpe : c'est la seule grandeur pour laquelle le carnet ait un critère de significativité
posé d'avance. Le Sharpe par décision est imprimé à titre descriptif — avec huit points de
décision et quatre graines, il n'a pas la puissance nécessaire pour trancher, et le carnet ne
s'en sert pas pour conclure.

**Le coût ensuite.** 288 appels pour la campagne — **295** au compteur avec les 7 appels de
démonstration — pour comparer deux protocoles sur huit dates : c'est le prix effectif du cadre,
et il est hors de portée d'un usage quotidien sur un univers large. Un cadre
multi-agents se justifie par un gain **mesurable**, jamais par son élégance. Ce prix se paie
aussi à chaque ré-exécution, avec un résultat qui change : la section 10 montre comment le
journal des appels le ramène à zéro pour rejouer une exécution donnée.

**La mémoire enfin.** Le carnet ne l'isole pas : il n'y a pas de troisième bras « cadre sans
mémoire ». Ce qui est montré, c'est qu'elle est **structurellement vide au premier point** et
qu'elle ne contient que des décisions soldées — deux propriétés nécessaires, pas suffisantes.
Isoler son effet demande un bras supplémentaire, laissé en exercice.

## 10. Journal, rejeu et noyau de risque

La campagne de la section 7 se paie en appels au modèle. Trois questions viennent ensuite, et aucune ne devrait coûter une seconde campagne.

1. **Le résultat se reproduit-il sans le modèle ?** Chaque appel de la campagne est inscrit, au moment où il part, dans un journal JSONL écrit hors du dépôt : rôle, empreinte du contenu exact envoyé, graine, réponse brute, et les champs que le code en a tirés (décision, prévision, réponse parsée ou non). Le **rejeu strict** relit ce seul fichier et refait toute la campagne — analystes, débat, PM, baseline, réflexion, mémoire — sans aucun appel. Une clé absente l'arrête : il ne devine jamais.
2. **Que change une phrase de consigne ?** Le **rejeu mixte** sert ce que le journal connaît et n'appelle le modèle que pour ce qui diffère. On modifie une seule consigne, celle du Bear, qui doit désormais *nommer le niveau de prix qui invaliderait son scénario*. Ce qui est en amont (analystes, baseline, premier message du Bull) reste servi par le journal ; ce qui est en aval (la suite du débat, le PM, la réflexion, puis, par la mémoire, les décisions suivantes) part au modèle. Plafond annoncé : 5 rappels par point, soit 160 pour la campagne.
3. **Que vaut la position implicite une fois passée par un noyau de risque ?** Le PM décide d'une direction, pas d'une taille (section 12). On greffe sur ses décisions, sans aucun appel, le noyau déterministe de `QC-Py-45` :

| Bras | `LONG` | `SHORT` | Stop à 2 ATR(14) | Coupe-circuit à −3 % |
|---|---|---|---|---|
| Sans noyau | 1 | −1 | non | non |
| Sans vente à découvert | 1 | 0 | non | non |
| Plafond 0,90 | 0,90 | 0 | non | non |
| Noyau sans coupe-circuit | 0,90 | 0 | oui | non |
| Noyau complet | 0,90 | 0 | oui | oui |

La position est prise à la clôture du point de décision et soldée à la clôture cinq séances plus tard. Le stop est posé à `clôture − 2 × ATR(14)` ; touché par le plus bas d'une séance, il est exécuté au plus bas de l'ouverture et du stop, et la position reste plate jusqu'à l'horizon. Le coupe-circuit liquide à la clôture une séance de portefeuille à −3 % ou pire, et la position est reprise à l'ouverture suivante si l'horizon n'est pas atteint. Les frais restent de 10 points de base par côté, sur le notionnel échangé. Pas de vente à découvert : le noyau n'emprunte pas.

Le verdict compare **Noyau complet** à **Sans noyau** avec la règle §C de la section 8, sur la perte `−pnl`. Tout ce protocole — nombre d'appels, consigne modifiée, correspondance décision → poids, cinq bras, critère — a été fixé avant la première mesure.

In [13]:
# A. Rejeu strict : toute la campagne refaite depuis le seul fichier journal, sans appel.
lignes_j = [json.loads(l) for l in FICHIER_JOURNAL.read_text(encoding="utf-8").splitlines()
            if l.strip()]
print("journal relu    : %s, %d lignes" % (FICHIER_JOURNAL.name, len(lignes_j)))
print("lignes par role :", dict(sorted(Counter(l["role"] for l in lignes_j).items())))

avant = APPELS["n"]
debut = time.time()
JOURNAL.ouvrir("rejouer", None, lignes_j)
res_rejeu = await campagne(SEEDS, CTX, trace=False)
JOURNAL.fermer()
appels_rejeu = APPELS["n"] - avant
assert appels_rejeu == 0, "le rejeu strict a appele le modele"

ecarts = {col: int((res_rejeu[col] != res[col]).sum()) for col in res_rejeu.columns}
CHAMPS = ("parse", "decision", "prevision_pct")
parsees = [l for l in lignes_j if "parse" in l]
divergentes = [l for l in parsees
               if JOURNAL.notes.get((l["cle"], l["occurrence"])) != {k: l[k] for k in CHAMPS}]

print("\nappels au modele : %d (duree %.1f s)" % (appels_rejeu, time.time() - debut))
print("reponses servies : %d / %d lignes du journal" % (JOURNAL.servis(), len(lignes_j)))
print("champs de la campagne compares : %d lignes x %d colonnes, %d ecart(s)"
      % (len(res_rejeu), len(ecarts), sum(ecarts.values())))
print("champs parses compares         : %d lignes du journal, %d divergente(s)"
      % (len(parsees), len(divergentes)))

journal relu    : journal_campagne.jsonl, 288 lignes
lignes par role : {'baseline': 32, 'debat:Bear': 64, 'debat:Bull': 64, 'momentum': 32, 'pm': 32, 'reflexion': 32, 'risque': 32}



appels au modele : 0 (duree 0.0 s)
reponses servies : 288 / 288 lignes du journal
champs de la campagne compares : 32 lignes x 13 colonnes, 0 ecart(s)
champs parses compares         : 128 lignes du journal, 0 divergente(s)


### Lecture — le rejeu strict, ou une exécution devenue un objet

**Le journal a exactement la forme du plan.** chaque appel y a sa ligne, et la répartition par rôle se recompose à la main : par décision, deux analystes (momentum, risque), quatre messages de débat (deux du Bull, deux du Bear), le PM, la baseline et la réflexion, soit 9 appels ; 32 décisions × 9 = 288. Une ligne manquante ou surnuméraire se verrait ici avant toute mesure.

**Le rejeu ne touche pas au modèle et retrouve tout.** `appels au modele : 0`, `288 / 288` réponses servies, et deux comparaisons à zéro écart :

- les 13 colonnes du tableau de campagne (prévisions, décisions, pertes, biais des analystes, mémoire) sont identiques sur toutes les lignes : toute la chaîne en aval des réponses, y compris la mémoire, est déterministe ;
- les 128 réponses à parser (PM, baseline, analystes) donnent les mêmes champs qu'à l'enregistrement : le parsing ne dépend que du texte reçu.

**Pourquoi c'est la pièce qui manquait.** Quatre exécutions successives de ce même code ont rendu des ratios `edge/sigma` de **+0,45, +0,63, +2,41 et −0,45** au verdict de la section 8. Les graines fixent le plan d'expérience, pas les réponses du modèle, qui varient d'un appel à l'autre. Une de ces quatre exécutions (+2,41, Diebold-Mariano p = 0,054) est passée à un cheveu du seuil §C : publiée seule, elle aurait raconté une autre histoire. Sans journal, un lecteur ne peut pas savoir laquelle de ces exécutions un carnet rapporte, ni la refaire ; avec lui, l'exécution devient un fichier que chacun rejoue à coût nul, et le verdict imprimé devient vérifiable.

La leçon dépasse ce carnet : dans une expérience qui appelle un LLM, la variance entre exécutions s'ajoute à la variance entre graines, et quatre graines ne la mesurent pas. Une conclusion se tire de plusieurs exécutions, ou d'une exécution journalisée qu'on peut rejouer.

In [14]:
# B. Rejeu mixte : une seule phrase change, la consigne du Bear.
CONSIGNE_BEAR_MODIFIEE = CONSIGNES["Bear"].replace(
    "a 5 seances.", "a 5 seances et nommer le niveau de prix qui invaliderait ton scenario.")
assert CONSIGNE_BEAR_MODIFIEE != CONSIGNES["Bear"]
print("consigne Bear modifiee :", CONSIGNE_BEAR_MODIFIEE)

PLAFOND_RAPPELS = 5 * len(SEEDS) * len(CTX)      # Bear x2, Bull au tour 2, PM, reflexion
FICHIER_BEAR = DOSSIER_JOURNAL / "journal_rejeu_bear.jsonl"
avant = APPELS["n"]
debut = time.time()
JOURNAL.ouvrir("mixte", FICHIER_BEAR, lignes_j)
res_bear = await campagne(SEEDS, CTX, consignes={"Bear": CONSIGNE_BEAR_MODIFIEE}, trace=False)
JOURNAL.fermer()
rappels = APPELS["n"] - avant

par_role = (pd.Series(JOURNAL.stats).unstack(fill_value=0)
            .reindex(columns=["journal", "appel"], fill_value=0))
print("\nservis par le journal / rappeles au modele, par role :")
print(par_role.to_string())
print("\nrappels : %d (plafond annonce %d, campagne complete %d) | duree %.1f min"
      % (rappels, PLAFOND_RAPPELS, annonce, (time.time() - debut) / 60.0))
print("journal  : %s, %d lignes inscrites, %.0f Ko (hors depot)"
      % (FICHIER_BEAR.name, len(JOURNAL.lignes), FICHIER_BEAR.stat().st_size / 1024))

change = res_bear["dec_pm"] != res["dec_pm"]
print("\ndecisions du PM qui changent : %d / %d" % (change.sum(), len(res)))
print("par graine :", {int(g): int(v) for g, v in change.groupby(res["graine"]).sum().items()})
print("temoins : baseline %d ecart(s), analystes %d ecart(s)"
      % ((res_bear["dec_bl"] != res["dec_bl"]).sum(),
         ((res_bear["biais_mom"] != res["biais_mom"])
          | (res_bear["biais_ris"] != res["biais_ris"])).sum()))
print(pd.crosstab(res["dec_pm"], res_bear["dec_pm"],
                  rownames=["campagne"], colnames=["Bear modifie"]))

consigne Bear modifiee : Tu defends le scenario baissier. Deux phrases maximum. Tu dois citer un chiffre du contexte et avancer une prevision chiffree a 5 seances et nommer le niveau de prix qui invaliderait ton scenario.



servis par le journal / rappeles au modele, par role :
            journal  appel
baseline         32      0
debat:Bear        0     64
debat:Bull       32     32
momentum         32      0
pm                0     32
reflexion         6     26
risque           32      0

rappels : 154 (plafond annonce 160, campagne complete 288) | duree 5.9 min
journal  : journal_rejeu_bear.jsonl, 154 lignes inscrites, 318 Ko (hors depot)

decisions du PM qui changent : 10 / 32
par graine : {0: 4, 1: 1, 7: 3, 42: 2}
temoins : baseline 0 ecart(s), analystes 0 ecart(s)
Bear modifie  FLAT  LONG  SHORT
campagne                       
FLAT             7     3      3
LONG             1    12      0
SHORT            3     0      3


### Lecture — une phrase de consigne, 154 appels, 10 décisions déplacées

**Le partage servi / rappelé suit le graphe de dépendances**, rôle par rôle :

- ce qui est **en amont** du Bear est servi en entier par le journal : analystes momentum et risque, baseline (`32 / 0` chacun) ;
- le **Bull** est coupé en deux : son premier message, écrit avant que le Bear parle, est servi (`32`) ; le second, qui répond au Bear modifié, est rappelé (`32`) ;
- le **Bear** et le **PM** sont rappelés en entier (`0 / 64`, `0 / 32`) ;
- la **réflexion** n'est servie que 6 fois sur 32 : quand le PM rappelé reproduit à l'identique ce qu'elle lit. Les 26 autres changent, et par la mémoire, elles changent les décisions suivantes.

Coût : `rappels : 154 (plafond annonce 160)`, en 5,9 min, contre 288 appels et 11,4 min pour la campagne complète. Le plafond posé d'avance tient.

**Le résultat** : `decisions du PM qui changent : 10 / 32`, réparties sur les quatre graines (4, 1, 3, 2), et les témoins sont intacts (`baseline 0 ecart(s), analystes 0 ecart(s)`) : le rejeu n'a rien touché en dehors de la chaîne modifiée. Le tableau croisé dit le sens des changements : parmi les 13 FLAT de la campagne, 3 deviennent LONG et 3 SHORT ; 3 des 6 SHORT deviennent FLAT ; un seul LONG devient FLAT. Les changements vont dans les deux sens : la consigne ne rend pas le PM plus baissier, le nombre de SHORT reste à 6, et le solde net est un glissement de FLAT vers LONG (13, 13, 6 devient 11, 15, 6).

**Ce que ce rejeu ne permet pas de conclure.** Un PM rappelé avec une consigne **inchangée** ne rendrait pas non plus exactement les mêmes décisions : le modèle n'est pas déterministe. Une partie des 10 changements peut donc venir du seul tirage. Le témoin qui manque est un second rejeu qui **force** le rappel de la même chaîne (Bear, second message du Bull, PM, réflexion) avec la consigne d'origine. Il faut le forcer : la clé du journal est l'empreinte du contenu envoyé, et une consigne inchangée serait servie en entier, sans rien mesurer. La part des décisions qui changent dans ce témoin est le bruit de fond ; seul l'excès au-delà de ce bruit s'attribue à la phrase. Son coût est connu d'avance : le même nombre de rappels, environ 154.

In [15]:
# C. Greffon deterministe : le noyau de risque applique aux decisions du PM, sans appel.
KERNEL_MAX_WEIGHT = 0.90          # plafond d'exposition par position (constante de QC-Py-45)
KERNEL_ATR_MULT = 2.0             # stop a 2 ATR sous le prix d'entree
KERNEL_ATR_N = 14
KERNEL_CB_THRESHOLD = -3 / 100    # coupe-circuit : seance du portefeuille a -3 %
FRAIS = COST_BPS / 10000.0

ohlc = brut.loc[prix.index, ["Open", "High", "Low", "Close"]].astype(float)
O, H, L, C = (ohlc[k].to_numpy() for k in ("Open", "High", "Low", "Close"))
assert np.allclose(C, prix.to_numpy())
veille = np.r_[np.nan, C[:-1]]
tr = np.nanmax(np.vstack([H - L, np.abs(H - veille), np.abs(L - veille)]), axis=0)
ATR = pd.Series(tr).rolling(KERNEL_ATR_N).mean().to_numpy()   # connu a la cloture du jour
POS = {str(d.date()): k for k, d in enumerate(prix.index)}


def simuler(i, w, stop=False, coupe=False):
    # Position de poids w prise a la cloture i, soldee a la cloture i + HORIZON.
    # Rend (pnl nette de frais, stops touches, coupures).
    if w == 0:
        return 0.0, 0, 0
    pnl, jambes, n_stop, n_coupe = 0.0, 0, 0, 0
    entree, niveau, ref, etat = C[i], C[i] - KERNEL_ATR_MULT * ATR[i], C[i], "ouvert"
    for d in range(i + 1, i + HORIZON + 1):
        if etat == "attente":                     # reprise a l'ouverture apres une coupure
            entree, niveau, ref, etat = O[d], O[d] - KERNEL_ATR_MULT * ATR[d - 1], O[d], "ouvert"
        if stop and L[d] <= niveau:               # stop touche : plat jusqu'a l'horizon
            pnl += w * (min(O[d], niveau) / entree - 1.0)
            jambes, n_stop, etat = jambes + 1, n_stop + 1, "plat"
            break
        if coupe and w * (C[d] / ref - 1.0) <= KERNEL_CB_THRESHOLD:
            pnl += w * (C[d] / entree - 1.0)
            jambes, n_coupe = jambes + 1, n_coupe + 1
            etat = "attente" if d < i + HORIZON else "plat"
            continue
        ref = C[d]
    if etat == "ouvert":
        pnl += w * (C[i + HORIZON] / entree - 1.0)
        jambes += 1
    return pnl - FRAIS * 2.0 * abs(w) * jambes, n_stop, n_coupe


SANS_EMPRUNT = {"LONG": KERNEL_MAX_WEIGHT, "FLAT": 0.0, "SHORT": 0.0}
BRAS = [
    ("Sans noyau",               {"LONG": 1.0, "FLAT": 0.0, "SHORT": -1.0}, False, False),
    ("Sans vente a decouvert",   {"LONG": 1.0, "FLAT": 0.0, "SHORT": 0.0},  False, False),
    ("Plafond 0,90",             SANS_EMPRUNT, False, False),
    ("Noyau sans coupe-circuit", SANS_EMPRUNT, True,  False),
    ("Noyau complet",            SANS_EMPRUNT, True,  True),
]
positions = [POS[d] for d in res["date"]]
graines = res["graine"].to_numpy()


def max_dd(p):
    eq = np.cumprod(np.r_[1.0, 1.0 + np.asarray(p, dtype=float)])
    return float((eq / np.maximum.accumulate(eq) - 1.0).min())


SORTIES, lignes_bras = {}, []
for nom, poids, stop, coupe in BRAS:
    sims = [simuler(i, poids[dec], stop, coupe) for i, dec in zip(positions, res["dec_pm"])]
    p = np.array([s[0] for s in sims])
    expo = np.array([poids[dec] for dec in res["dec_pm"]])
    SORTIES[nom] = p
    lignes_bras.append({
        "bras": nom,
        "pnl moy. %": p.mean() * 100,
        "Sharpe/decision": sharpe_par_point(p),
        "expo. nette moy.": expo.mean(),
        "maxDD moy./graine %": np.mean([max_dd(p[graines == g]) for g in SEEDS]) * 100,
        "stops": sum(s[1] for s in sims),
        "coupures": sum(s[2] for s in sims),
    })

assert np.allclose(SORTIES["Sans noyau"], p_pm), "le bras sans noyau doit redonner p_pm"
print("controle : 'Sans noyau' redonne la pnl de la lecture economique (ecart max %.1e)"
      % np.abs(SORTIES["Sans noyau"] - p_pm).max())
print("decisions du PM :", {k: int(v) for k, v in res["dec_pm"].value_counts().items()})
print()
print(pd.DataFrame(lignes_bras).set_index("bras").round(3).to_string())

# Controle positif : une position LONG hypothetique a 0,90 sur chacune des 8 dates.
# Il dit si le stop et le coupe-circuit auraient eu quelque chose a couper, et ou etait le PM.
temoin = []
for i, c in zip(INDICES, CTX):
    p_nu, _, _ = simuler(i, KERNEL_MAX_WEIGHT)
    p_k, s_k, c_k = simuler(i, KERNEL_MAX_WEIGHT, stop=True, coupe=True)
    decs = Counter(res.loc[res["date"] == c["date"], "dec_pm"])
    temoin.append({"date": c["date"],
                   "LONG 0,90 nu %": p_nu * 100, "avec noyau %": p_k * 100,
                   "stop": s_k, "coupures": c_k,
                   "PM L/F/S": "%d/%d/%d" % (decs["LONG"], decs["FLAT"], decs["SHORT"])})
print("\ncontrole positif : LONG hypothetique a 0,90 sur chaque date (PM : decisions des 4 graines)")
print(pd.DataFrame(temoin).set_index("date").round(3).to_string())

controle : 'Sans noyau' redonne la pnl de la lecture economique (ecart max 0.0e+00)
decisions du PM : {'LONG': 13, 'FLAT': 13, 'SHORT': 6}

                          pnl moy. %  Sharpe/decision  expo. nette moy.  maxDD moy./graine %  stops  coupures
bras                                                                                                         
Sans noyau                    -0.017           -0.009             0.219               -7.589      0         0
Sans vente a decouvert         0.017            0.011             0.406               -5.203      0         0
Plafond 0,90                   0.015            0.011             0.366               -4.688      0         0
Noyau sans coupe-circuit       0.032            0.023             0.366               -4.559      1         0
Noyau complet                  0.032            0.023             0.366               -4.559      1         1

controle positif : LONG hypothetique a 0,90 sur chaque date (PM : decisions des 4 graines

### Lecture — cinq bras, un témoin, un contrôle positif

**Le témoin d'abord** : `'Sans noyau' redonne la pnl de la lecture economique (ecart max 0.0e+00)`. Le greffon rejoue exactement les décisions et la pnl de la section 8 avant d'y toucher : les écarts entre bras viennent du noyau, pas d'une différence de câblage.

**Les bras ensuite, une contrainte à la fois** :

- **Retirer la vente à découvert** fait le plus gros du chemin : la pnl moyenne passe de `-0.017` à `+0.017` %, le drawdown moyen par graine de `-7.589` à `-5.203` %. Les 6 décisions SHORT du PM coûtaient sur cette fenêtre.
- **Plafonner à 0,90** réduit l'exposition (`0.406` → `0.366`) et le drawdown (`-4.688` %) pour une pnl quasi inchangée.
- **Le stop à 2 ATR** mord une fois (`stops 1`) et porte la pnl à `+0.032` %.
- **Le coupe-circuit** se déclenche une fois (`coupures 1`) sans effet visible à la précision affichée.

**Le contrôle positif enfin.** Sur les 32 décisions, le stop et le coupe-circuit n'ont presque rien eu à faire : il faut vérifier qu'ils fonctionnent quand l'occasion se présente. Le tableau prend une position LONG hypothétique à 0,90 sur chacune des huit dates :

- le **2024-09-02**, la séance de baisse touche le stop : `-7.751` % sans noyau, `-7.228` % avec. Le PM n'était LONG que sur une graine ce jour-là (`1/2/1`) : c'est le stop de la campagne ;
- le **2024-05-05** et le **2025-08-28**, le coupe-circuit se déclenche, et le second cas **coûte** (`-1.255` → `-1.349` %) : liquider à la clôture puis reprendre à l'ouverture suivante ajoute un aller-retour de frais et peut racheter plus haut. Un coupe-circuit est une assurance, et une assurance a un prix.

Le noyau se comporte comme spécifié. Sur cet échantillon, ce qui améliore la pnl n'est pas la protection contre les queues, mais une règle de portefeuille : ne pas emprunter pour vendre.

In [16]:
# D. Verdict §C du greffon : Noyau complet contre Sans noyau, pertes appariees.
perte_sans = -SORTIES["Sans noyau"]
perte_noyau = -SORTIES["Noyau complet"]
d_noyau = perte_noyau - perte_sans               # negatif = le noyau perd moins

edge_g = pd.Series(-d_noyau).groupby(graines).mean()
edge_k = float(edge_g.mean())
sigma_k = float(edge_g.std(ddof=1))
if sigma_k > 0:
    ratio_k = edge_k / sigma_k
else:
    ratio_k = 0.0 if edge_k == 0 else math.copysign(float("inf"), edge_k)
_, stat_k, p_k = diebold_mariano(d_noyau)

print("edge par graine (points de %, positif = le noyau perd moins) :")
for g, e in edge_g.items():
    _, s_g, p_g = diebold_mariano(d_noyau[graines == g])
    print("  graine %-3d : %+.3f   (DM descriptif, n=%d : stat %+.2f, p = %.3f)"
          % (g, e * 100, (graines == g).sum(), s_g, p_g))
print("\nedge moyen          : %+.4f pt" % (edge_k * 100))
print("sigma inter-graines : %.4f pt (n=%d)" % (sigma_k * 100, len(SEEDS)))
print("edge / sigma        : %+.2f  (seuil §C : >= 2)" % ratio_k)
print("Diebold-Mariano     : stat %+.3f, p = %.4f (%d paires, h=1)" % (stat_k, p_k, len(d_noyau)))

if ratio_k >= 2.0 and p_k < 0.05 and stat_k < 0:
    verdict_noyau = "BEATS"
elif ratio_k <= -2.0 and p_k < 0.05 and stat_k > 0:
    verdict_noyau = "NO BEATS"
else:
    verdict_noyau = "INCONCLUSIVE"
print("\nVERDICT NOYAU :", verdict_noyau)

print("\nrapport de biais, moyennes sur les %d decisions :" % len(res))
for nom, poids, _, _ in (BRAS[0], BRAS[-1]):
    expo = np.array([poids[dec] for dec in res["dec_pm"]])
    print("  %-14s : exposition nette %+.3f | pnl %+.3f%% | d moyen %+.3f pt"
          % (nom, expo.mean(), SORTIES[nom].mean() * 100,
             0.0 if nom == "Sans noyau" else d_noyau.mean() * 100))

edge par graine (points de %, positif = le noyau perd moins) :
  graine 0   : -0.171   (DM descriptif, n=8 : stat +1.44, p = 0.150)
  graine 1   : +0.633   (DM descriptif, n=8 : stat -1.04, p = 0.297)
  graine 7   : +0.127   (DM descriptif, n=8 : stat -0.73, p = 0.463)
  graine 42  : -0.394   (DM descriptif, n=8 : stat +0.45, p = 0.655)

edge moyen          : +0.0489 pt
sigma inter-graines : 0.4442 pt (n=4)
edge / sigma        : +0.11  (seuil §C : >= 2)
Diebold-Mariano     : stat -0.168, p = 0.8662 (32 paires, h=1)

VERDICT NOYAU : INCONCLUSIVE

rapport de biais, moyennes sur les 32 decisions :
  Sans noyau     : exposition nette +0.219 | pnl -0.017% | d moyen +0.000 pt
  Noyau complet  : exposition nette +0.366 | pnl +0.032% | d moyen -0.049 pt


### Lecture — le verdict du noyau : un effet descriptif, aucune preuve

La sortie applique au noyau la même règle que la section 8, sur la perte `−pnl` appariée décision par décision :

1. **Quatre edge par graine de signes opposés** : `-0.171 / +0.633 / +0.127 / -0.394`. Deux graines où le noyau perd moins, deux où il perd plus ; les tests descriptifs par graine (n=8) restent tous loin de 0,05.
2. **`edge +0.0489, sigma 0.4442` → `edge/sigma +0.11`**, et **`Diebold-Mariano stat -0.168, p = 0.8662`**. Le signe va dans le sens du noyau, l'ampleur est un dixième de la dispersion : `VERDICT NOYAU : INCONCLUSIVE`.
3. **Le rapport de biais dit d'où vient l'effet.** L'exposition nette moyenne passe de `+0.219` à `+0.366` et la perte moyenne baisse de `0.049` pt : c'est le retrait des positions courtes, visible dans le tableau des bras, pas le stop ni le coupe-circuit, qui ne se sont déclenchés qu'une fois chacun sur 32 décisions.

Ce résultat est attendu, et il est instructif. Un noyau de risque agit sur les **queues** de la distribution — séances violentes, enchaînements de pertes — et huit dates en visitent très peu : sur cet échantillon, sa valeur n'est pas mesurable, ni dans un sens ni dans l'autre. Le ratio ne se stabilise pas davantage d'une exécution à l'autre : sur trois exécutions de la campagne, le greffon a donné +0,57, −0,78 puis +0,11, toujours INCONCLUSIVE. Juger un noyau de risque demande un backtest long sur des données de marché, ce que fait `QC-Py-45` ; ici, il sert à montrer qu'une décision de direction ne devient une position qu'après une règle de taille, et que cette règle se teste séparément, à coût nul, sur des décisions déjà journalisées.

## 11. Exercices

**Exercice 1 — un troisième analyste.** Ajoutez un analyste de flux qui ne regarde que
`cinq_jours` (les cinq derniers rendements) et mesurez si le débat change de camp plus
souvent. Le coût augmente de 8 appels par graine.

**Repère de complétion (exercice 1).** `analyste_flux(c, graine)` doit appeler `analyste(SYST_FLUX, c, graine)` et rendre la `Reponse` obtenue — le système `SYST_FLUX` est déjà écrit dans la cellule, le TODO est l'appel. Quatre repères de vérification :

1. **La réponse doit satisfaire le `SCHEMA`** collé à `SYST_FLUX` : même structure que les deux analystes existants (momentum et risque). C'est ce qui la rend parsable par le cadre — les compteurs de la section 7 séparaient déjà « non parsees: analystes 6/64 », un compte qui a varié de 6 à 14 sur 64 d'une exécution à l'autre : un troisième analyste fragile ajoute sa propre fuite au lieu d'enrichir le débat. Testez sur 4-5 appels avant de lancer la campagne.
2. **Le coût annoncé est vérifiable au compteur.** L'énoncé dit « +8 appels par graine » : la campagne mesurait 72 appels par graine (incréments 79→151→223→295), elle doit passer à 80 — soit 320 appels de campagne au lieu de 288, et 327 au compteur final au lieu de 295, temps proportionnel (~12,7 min au rythme mesuré de 11,4 min). Si le compteur final diffère, l'analyste a été appelé ailleurs qu'aux points de décision.
3. **Le contrat de « ne regarder que `cinq_jours` ».** L'analyste de flux est défini par sa restriction d'information : son prompt ne doit voir que les cinq derniers rendements, pas `ret20` ni `vol20`. C'est le même principe d'information asymétrique qui sépare déjà le Bull (dernière valeur) du Bear (historique) — chaque rôle du débat est une hypothèse sur *quelle* fenêtre d'information suffit à décider.
4. **La mesure attendue est un comptage de changements de camp.** « Le débat change-t-il de camp plus souvent ? » se mesure sur les décisions du cadre : comparez la distribution des décisions à 2 analystes (mesurée, section 8) contre 3 — pas sur les opinions des analystes eux-mêmes. Un troisième avis ne compte que s'il déplace l'arbitre.

**Exercice 2 — la mémoire par similarité.** Remplacez `Memoire.resume` (les trois dernières
entrées) par une mémoire qui rend les trois entrées dont le contexte est le plus **proche** de
la décision courante (distance euclidienne sur `ret20`, `vol20`, `zscore20`, normalisés). La
question devient : la mémoire aide-t-elle parce qu'elle est récente, ou parce qu'elle est
pertinente ?

**Repère de complétion (exercice 2).** `resume_par_similarite(memoire, c, k=3)` doit remplacer la règle « trois dernières entrées » de `Memoire.resume` par « trois entrées **au contexte le plus proche** ». Quatre repères :

1. **La normalisation d'abord, la distance ensuite.** `ret20`, `vol20` et `zscore20` n'ont pas la même échelle (un rendement en pourcents contre un z-score centré) : sans normalisation sur les entrées soldées, la distance euclidienne est dominée par la feature de plus grand ordre de grandeur et la similarité ne fait que répéter cette feature. Normaliser chaque dimension sur l'ensemble des entrées de la mémoire, puis calculer la distance au contexte courant `c` normalisé avec les mêmes paramètres — sinon la comparaison mélange deux échelles.
2. **Les premiers appels doivent dégrader proprement.** La mémoire part vide (`memoire vide -> 0 entree(s)` en section 5) : au premier point de décision il n'y a rien à trier. La fonction doit rendre la chaîne de repli (moins de `k` entrées soldées → rendre ce qui existe), jamais lever d'exception sur une mémoire vide.
3. **La sortie reste une chaîne pour le prompt.** `Memoire.resume` nourrit le prompt du cadre — la valeur de retour doit rester un texte insérable, pas une liste d'objets. Le stub rend `"(exercice non complete)"`, qui est précisément un texte neutre pour ce prompt.
4. **La mesure de l'effet existe déjà.** La question « récent ou pertinent ? » se tranche avec les compteurs du carnet : le taux d'accord entre analystes (34,4 % (11/32) en section 8) et l'edge par graine de la section 8 sont les deux indicateurs à comparer entre la mémoire par récence (mesurée) et la mémoire par similarité (votre variante). Si le taux d'accord ne bouge pas mais que l'edge change, la mémoire agissait par l'information de contexte ; si rien ne bouge, elle n'agissait pas du tout — réponse tout aussi instructive. Un écart ne compte que s'il dépasse la variation d'une exécution à l'autre (section 10) ; un rejeu mixte ne rappelle que ce qui dépend de la mémoire, et réduit d'autant le coût de la comparaison.

**Exercice 3 — l'horizon.** Faites varier `HORIZON` (1, 5, 20 séances). Un cadre qui gagne à
5 séances gagne-t-il encore à 20 ? Le verdict porte-t-il sur le protocole ou sur une
coïncidence d'horizon ?

**Repère de complétion (exercice 3).** `campage_horizon(h)` doit relancer la campagne complète pour un horizon de détention `h` pris dans {1, 5, 20}. Trois repères avant d'écrire :

1. **La marge de soldage dépend de `HORIZON`** — l'avertissement de l'énoncé n'est pas décoratif. Les indices de décision doivent rester à distance d'au moins `h` séances de la fin de l'historique (3 042 séances) : à horizon 20 avec 8 points, il faut vérifier que le dernier point laisse 20 séances jouables. La formule de construction d'`INDICES` du carnet donne le modèle exact — c'est elle qu'il faut paramétrer, pas recopier.
2. **Le verdict doit être rejoué par l'organe du §C** (edge, sigma, edge/sigma, DM, verdict INCONCLUSIVE) — pas résumé à la main. Si votre fonction imprime un verdict sans les quatre nombres, elle ne permet pas la comparaison avec la campagne à horizon 5 (`edge -0.0848, sigma 0.1874, edge/sigma -0.45, DM p = 0.4485`), qui est la référence du contraste. Une différence entre horizons ne compte que si elle dépasse la variation d'une exécution à l'autre au même horizon : la section 10 a vu le ratio aller de −0,45 à +2,41 sans rien changer au code.
3. **Le coût est linéaire en points × graines** : à 8 points × 4 graines, chaque horizon complet consomme ~288 appels (le §9 compte 288 pour la campagne, 295 au compteur avec les 7 appels de démonstration). Trois horizons = trois campagnes ; le temps de 11,4 min mesuré en §7 borne l'attente par horizon.

La question de fond posée par l'énoncé — « le verdict porte-t-il sur le protocole ou sur une coïncidence d'horizon ? » — est la bonne question de robustesse : si l'edge du cadre s'inverse à horizon 20, ce que la section 8 a mesuré n'est pas une propriété du cadre mais de la fenêtre de détention choisie. Un cadre qui ne gagne qu'à 5 séances n'a pas démontré une capacité de décision, il a démontré une sensibilité de paramètre.

In [17]:
# --- Exercice 1 : un troisieme analyste (flux) --------------------------------
SYST_FLUX = ("Tu es analyste de flux. Tu ne regardes que les cinq derniers rendements et tu "
             "juges la persistance court terme. " + SCHEMA)

def analyste_flux(c, graine):
    # TODO etudiant : appeler `analyste(SYST_FLUX, c, graine)` et renvoyer la Reponse.
    print("Exercice a completer : analyste de flux")
    return None

print("Exercice 1 pret a completer.")

Exercice 1 pret a completer.


In [18]:
# --- Exercice 2 : memoire par similarite de contexte -------------------------

def resume_par_similarite(memoire, c, k=3):
    # TODO etudiant : normaliser ret20, vol20, zscore20 sur les entrees soldees,
    # calculer la distance au contexte courant `c`, rendre les k plus proches.
    print("Exercice a completer : memoire par similarite")
    return "(exercice non complete)"

print("Exercice 2 pret a completer.")

Exercice 2 pret a completer.


In [19]:
# --- Exercice 3 : l'horizon de detention -------------------------------------

def campage_horizon(h):
    # TODO etudiant : reconstruire INDICES et CTX avec cet horizon, relancer la campagne,
    # imprimer le verdict §C. Attention : la marge de soldage depend de HORIZON.
    print("Exercice a completer : campagne a horizon %d" % h)

print("Exercice 3 pret a completer.")

Exercice 3 pret a completer.


## 12. Ce que ce carnet ne fait pas

- **Aucune donnée textuelle.** Les analystes de TradingAgents lisent des actualités, des
  états financiers et des réseaux sociaux. Ici, tout est numérique : le carnet mesure un
  protocole de décision, pas une capacité de lecture.
- **Un seul actif, un seul marché.** Les conclusions ne se transportent pas à un portefeuille
  multi-actifs ni à des actions.
- **Pas d'équipe de risque dans le débat.** Le PM décide d'une direction et d'une amplitude,
  pas d'une taille de position. La section 10 greffe après coup un noyau de risque
  déterministe sur ses décisions ; le modèle ne le voit jamais, et ne peut donc pas en tenir
  compte en décidant.
- **La mémoire n'est pas isolée.** Voir la section 9 : un troisième bras serait nécessaire.
- **Huit points de décision.** C'est peu. Le Diebold-Mariano porte sur 32 paires, ce qui suffit
  à détecter un écart franc, pas un écart fin ; un verdict `INCONCLUSIVE` doit se lire comme
  « pas de preuve », jamais comme « pas d'effet ».
- **Une exécution n'est pas l'expérience.** Les réponses du modèle varient d'un appel à
  l'autre à graine fixée : le verdict imprimé est celui d'une exécution journalisée, et la
  section 10 montre qu'une autre exécution du même code peut en rendre un autre.

## Références

- Xiao, Sun, Luo & Wang — *TradingAgents: Multi-Agents LLM Financial Trading Framework*,
  arXiv:2412.20138 (v7, 2025-06-03). PDF archivé au gisement bibliographique du dépôt.
- `GenAI/SemanticKernel/03-SemanticKernel-Agents.ipynb` — organe `AgentGroupChat`.
- `QC-Py-42-Alpha-Mining-Evolution.ipynb` — doctrine §C, Diebold-Mariano local, coûts.
- `QC-Py-45-Noyau-Risque-Deterministe.ipynb` — noyau de risque déterministe greffé en
  section 10 (plafond 0,90, stop à 2 ATR, coupe-circuit).

[<< Sommaire QC](../README.md) | [Précédent : QC-Py-43-Diffusion-Series-Synthetiques <<](./QC-Py-43-Diffusion-Series-Synthetiques.ipynb) | [Suivant : QC-Py-45-Noyau-Risque-Deterministe >>](./QC-Py-45-Noyau-Risque-Deterministe.ipynb)